Import libraries (pytorch, pandas, etc)

In [1]:
!pip install pytorch_fid
!pip install torchmetrics[image] torch-fidelity

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import math
from pytorch_fid.fid_score import calculate_fid_given_paths
from torchmetrics.image.inception import InceptionScore
import torch.utils.data as data
import os

Get MNIST dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
T = 1000
s = 8e-3

# Standard cosine beta schedule (corrected step division)
steps = torch.linspace(
    0, T, T + 1,
    dtype=torch.float64,
    device=device
)
alpha_bar = torch.cos(((steps / T + s) / (1 + s)) * math.pi / 2) ** 2
alpha_bar = alpha_bar / alpha_bar[0]

betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
betas = betas.clamp(min=1e-5, max=0.999).float()

alphas = 1.0 - betas
alpha_time_t = torch.cumprod(alphas, dim=0)

In [7]:
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)

    sqrt_alpha_time_t = torch.sqrt(alpha_time_t[t])
    sqrt_one_minus_alpha_time_t = torch.sqrt(1 - alpha_time_t[t])

    # Reshape so the values can be multiplied across the entire picture
    sqrt_alpha_time_t = sqrt_alpha_time_t[:, None, None, None]
    sqrt_one_minus_alpha_time_t = sqrt_one_minus_alpha_time_t[:, None, None, None]

    xt = (
        sqrt_alpha_time_t * x0
        + sqrt_one_minus_alpha_time_t * noise
    )

    return xt, noise

In [9]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device

        half_dim = self.dim // 2

        emb_scale = math.log(10000) / (half_dim - 1)

        emb = torch.exp(
            torch.arange(half_dim, device=device) * -emb_scale
        )

        emb = t.float()[:, None] * emb[None, :]

        emb = torch.cat(
            [torch.sin(emb), torch.cos(emb)],
            dim=1
        )

        return emb

In [10]:
class block(nn.Module):
    def __init__(self, in_channels, out_channels, num_groups=8):
        super().__init__()
        self.normalize = nn.GroupNorm(num_groups, in_channels)
        self.convolute = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act_func = nn.SiLU()

    def forward(self, x):
        x = self.normalize(x)
        x = self.act_func(x)
        x = self.convolute(x)
        return x


In [11]:
class resnetblock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.mlp = nn.Linear(time_dim, out_channels)
        self.block1 = block(in_channels, out_channels)
        self.block2 = block(out_channels, out_channels)

        if in_channels != out_channels:
            self.residual = nn.Conv2d(in_channels, out_channels, kernel_size = 1)
        else: self.residual = nn.Identity()

    def forward(self, x, time_emb):
        h = self.block1(x)
        time_emb = self.mlp(time_emb)
        time_emb = time_emb[:, :, None, None]
        h = h + time_emb
        h = self.block2(h)

        return h + self.residual(x)



In [12]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, time_dim=128):
        super().__init__()
        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )
        #initial convolution to get 64 channels from the base 1 channel since the pictures are greyscale
        self.input_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)

        #first downblock, convoluting and learning features while sending skip connection to upblocks
        self.downblock1 = resnetblock(base_channels, base_channels, time_dim)

        #first downsample, where number of channels increases for more feature learning and stride=2 to turn the 28 by 28 pictures into 14 by 14
        self.downsample1 = nn.Conv2d(base_channels, base_channels * 2, kernel_size = 4, stride=2, padding=1)

        self.downblock2 = resnetblock(base_channels * 2, base_channels * 2, time_dim)

        self.downsample2 = nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)

        self.bottleneck1 = resnetblock(base_channels * 4, base_channels * 4, time_dim)

        self.bottleneck2 = resnetblock(base_channels * 4, base_channels * 4, time_dim)

        #first upsample, reducing number of channels and increasing from 7 by 7 image to 14 by 14 using convtranspose2d
        self.upsample1 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size = 4, stride=2, padding=1)

        # Fix: input channels for upblock1 should be base_channels * 4 due to concatenation
        self.upblock1 = resnetblock(base_channels * 4, base_channels * 2, time_dim)

        self.upsample2 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)

        # Fix: input channels for upblock2 should be base_channels * 2 due to concatenation
        self.upblock2 = resnetblock(base_channels * 2, base_channels, time_dim)

        self.output = nn.Conv2d(base_channels, in_channels, kernel_size=1)

    def forward(self, x, t):
        t_emb = self.time_embedding(t)
        x = self.input_conv(x)
        d1 = self.downblock1(x, t_emb)
        x = self.downsample1(d1)
        d2 = self.downblock2(x, t_emb)
        x = self.downsample2(d2)
        x = self.bottleneck1(x, t_emb)
        x = self.bottleneck2(x, t_emb)
        x = self.upsample1(x)
        x = torch.cat([x, d2], dim=1)
        u1 = self.upblock1(x, t_emb)
        x = self.upsample2(u1)
        x = torch.cat([x, d1], dim=1)
        u2 = self.upblock2(x, t_emb)
        return self.output(u2)


In [ ]:
model = UNet().to(device)
num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"{num_params:,} parameters")

4,789,699 parameters


In [ ]:
PATH = "/checkpoints/ddpm_cifar_250.pth"
model.load_state_dict(PATH['model_state_dict'])

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


<All keys matched successfully>

Train from scratch if needed:

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

num_epochs = 250

loss_history = []

for epoch in range(num_epochs):

    print("epoch number: ", epoch)

    model.train()

    for x0, _ in train_loader:
        x0 = x0.to(device)

        batch_size = x0.shape[0]

        # Random timestep for each image
        t = torch.randint(
            0,
            T,
            (batch_size,), device=device
        )

        # Random Gaussian noise
        noise = torch.randn_like(x0)

        # Forward diffusion
        xt, _ = q_sample(
            x0,
            t,
            noise
        )


        # Predict the noise
        pred_noise = model(
            xt,
            t
        )

        # DDPM loss
        loss = F.mse_loss(
            pred_noise,
            noise
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        loss_history.append(loss.item())

epoch number:  0
epoch number:  1
epoch number:  2
epoch number:  3
epoch number:  4
epoch number:  5
epoch number:  6
epoch number:  7
epoch number:  8
epoch number:  9
epoch number:  10
epoch number:  11
epoch number:  12
epoch number:  13
epoch number:  14
epoch number:  15
epoch number:  16
epoch number:  17
epoch number:  18
epoch number:  19
epoch number:  20
epoch number:  21
epoch number:  22
epoch number:  23
epoch number:  24
epoch number:  25
epoch number:  26
epoch number:  27
epoch number:  28
epoch number:  29
epoch number:  30
epoch number:  31
epoch number:  32
epoch number:  33
epoch number:  34
epoch number:  35
epoch number:  36
epoch number:  37
epoch number:  38
epoch number:  39
epoch number:  40
epoch number:  41
epoch number:  42
epoch number:  43
epoch number:  44
epoch number:  45
epoch number:  46
epoch number:  47
epoch number:  48
epoch number:  49
epoch number:  50
epoch number:  51
epoch number:  52
epoch number:  53
epoch number:  54
epoch number:  55
ep

In [ ]:
torch.manual_seed(42)

Create sampler, to use trained model to generate images

In [15]:
@torch.no_grad()
def sample_ddpm(
    model,
    n_samples,
    image_size=32
):
    model.eval()

    x = torch.randn(
        n_samples,
        3, #in channels value
        image_size,
        image_size,
        device=device
    )

    for t_value in reversed(range(T)):

        t = torch.full(
            (n_samples,),
            t_value,
            device=device,
            dtype=torch.long
        )

        # Predict noise
        pred_noise = model(x, t)

        beta_t = betas[t_value]
        alpha_t = alphas[t_value]
        alpha_bar_t = alpha_time_t[t_value]

        # DDPM mean
        mean = (
            1 / torch.sqrt(alpha_t)
        ) * (
            x
            -
            (beta_t / torch.sqrt(1 - alpha_bar_t))
            * pred_noise
        )

        if t_value > 0:

            noise = torch.randn_like(x)

            variance = beta_t

            x = (
                mean
                + torch.sqrt(variance) * noise
            )

        else:
            x = mean

    return x

In [ ]:
import os
from PIL import Image

generated_dir = "./generated_cifar"
os.makedirs(generated_dir, exist_ok=True)

In [18]:

def batch_sample_gen(n_samples, batches, model, generated_dir):

    inception_score = InceptionScore(
        splits=12,
        normalize=True
    )

    os.makedirs(generated_dir, exist_ok=True)

    samples_per_batch = n_samples // batches

    for i in range(batches):

        print("Batch number:", i)

        samples = sample_ddpm(
            model,
            n_samples=samples_per_batch
        )

        # Convert [-1, 1] to [0, 1]
        samples = (samples.clamp(-1, 1) + 1) / 2

        # Update IS using normalized images
        is_samples = samples[0:100]
        inception_score.update(is_samples.to('cpu'))

        # Convert to uint8 for saving
        images = (samples * 255).round().to(torch.uint8)

        for j, img in enumerate(images):

            # [C, H, W] -> [H, W, C]
            img = img.permute(1, 2, 0).cpu().numpy()

            pic_no = i * samples_per_batch + j

            Image.fromarray(img).save(
                os.path.join(
                    generated_dir,
                    f"{pic_no:05d}.png"
                )
            )

    # Calculate IS once across all generated images
    is_mean, is_std = inception_score.compute()

    return is_mean.item(), is_std.item()


In [ ]:
is_mean, is_std = batch_sample_gen(n_samples=3000, batches=30, model=model, generated_dir=generated_dir)

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


Batch number: 0
Batch number: 1
Batch number: 2
Batch number: 3
Batch number: 4
Batch number: 5
Batch number: 6
Batch number: 7
Batch number: 8
Batch number: 9


In [ ]:
real_dir = "./real_cifar"
os.makedirs(real_dir, exist_ok=True)

counter = 0
n_samples = 3000

for images, _ in train_loader:
    for image in images:
        if counter >= n_samples:
            break

        # Convert [-1, 1] -> [0, 1] -> [0, 255]
        image = (image.clamp(-1, 1) + 1) / 2
        image = (image * 255).round().to(torch.uint8)

        # [C, H, W] -> [H, W, C]
        image = image.permute(1, 2, 0).cpu().numpy()

        Image.fromarray(image).save(
            os.path.join(real_dir, f"{counter:05d}.png")
        )

        counter += 1

    if counter >= n_samples:
        break

In [ ]:
fid_value = calculate_fid_given_paths(
    paths=['./real_cifar', './generated_cifar'],
    batch_size=100,
    device='cuda',
    dims=2048
)
print(f"FID score: {fid_value}")
print(f"IS score and stddev: {is_mean}, {is_std}")

Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:00<00:00, 169MB/s]
100%|██████████| 200/200 [00:47<00:00,  4.21it/s]


FID score: 28.10739689757611
